
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>



# Lab - Applying Agent Evaluation

**Overview**

This lab provides hands-on experience with MLflow's evaluation framework for generative AI agents. You will work with the NYC Taxi dataset to build evaluation workflows using both built-in judges (correctness, safety) and custom guideline judges.

In this lab, you will create evaluation datasets, configure multiple types of judges, and analyze evaluation results to understand agent performance. The lab emphasizes practical implementation of automated evaluation workflows that can be applied to real-world AI applications.

**Learning Objectives**

By the end of this lab, you will be able to:

- **Configure and execute** MLflow's built-in judges for correctness and safety assessment
- **Implement guideline judges** for custom business rule evaluation using natural language criteria
- **Create evaluation datasets** with proper structure for different judge types
- **Analyze evaluation results** using MLflow's tracing capabilities and result dataframes
- **Apply best practices** for separating evaluation models and managing configuration files


<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Prerequisites</strong>
<p style="margin: 8px 0 0 0; color: #333;"> This demo uses the agent created in <strong>01 - Agent Setup</strong>. Please ensure you have completed that notebook before proceeding.</p>
</div>
</div>
</div>


<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">That you might need to rerun the cells in the notebook if you receive an <strong>ValueError</strong> or <strong>MLflowException</strong> error regarding a datatype that was generated as a part of the agent's trace.</p>
</div>
</div>
</div>

## REQUIRED - SELECT SERVERLESS COMPUTE

Before executing cells in this notebook, attach the notebook to **Serverless compute**.

**NOTE:** This demo was tested on **Serverless (version 4)**.  
To confirm or change your Serverless version, see the Databricks documentation on Serverless dependencies.

### Compute Requirements

This course has been configured to run on Serverless compute. While classic compute may also work, testing has been performed on serverless.

**This demo was tested using version 4 of Serverless compute.** To ensure that you are using the correct version of Serverless, please [see this documentation on viewing and changing your notebook's Serverless version](https://docs.databricks.com/aws/en/compute/serverless/dependencies).

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Required - Select Serverless Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">You must attach this notebook to a Serverless compute resource before proceeding.</p>
</div>
</div>
</div>

### Classroom Setup

Run the following cell to configure your working environment for this course.

This setup will:
- Initialize the `DA` object (Databricks Academy helper)
- Configure your **default catalog** and **schema**
- Provision any supporting configuration needed for this demo

**NOTE:** The `DA` object is only available in Databricks Academy courses

In [0]:
%run ../Includes/Classroom-Setup-4

**Other Conventions:**

Throughout this Lab, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

## Part 1. Understanding the NYC Taxi Dataset and Agent Functions

### 1.1. Test Agent Functions

Let's test the agent's capabilities by calling it with different types of queries to understand how it uses the available functions.

In [0]:
def agent_payload(question: str):
    return [{"input": [{"role": "user", "content": question}]}]

In [0]:
## Test the agent with a query about trip information
## Use a query that would trigger the get_trip_info function
test_query_1 = <FILL_IN>
result_1 = agent.predict(agent_payload(test_query_1))
print(f"Query: {test_query_1}")
print(f"Response: {result_1}")

<details>
<summary style="cursor: pointer; padding: 10px; border: 1px solid #ccc; background-color: #f9f9f9; border-radius: 5px; font-weight: bold;">Click to view answer</summary>

<button onclick="copyAnswer1()">Copy to clipboard</button>

<pre id="answer-1" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
test_query_1 = "What's the average trip distance for trips from zip code 10001 to 10002?"
result_1 = agent.predict(agent_payload(test_query_1))
print(f"Query: {test_query_1}")
print(f"Response: {result_1}")
</code></pre>

</details>

<script>
function copyAnswer1() {
  const el = document.getElementById("answer-1");
  if (!el) return;

  const text = el.innerText;

  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

In [0]:
## Test the agent with a fare calculation query
## Use a query that would trigger the calculate_fare_estimate function
test_query_2 = <FILL_IN>
result_2 = agent.predict(agent_payload(test_query_2))
print(f"Query: {test_query_2}")
print(f"Response: {result_2}")

<details>
<summary style="cursor: pointer; padding: 10px; border: 1px solid #ccc; background-color: #f9f9f9; border-radius: 5px; font-weight: bold;">Click to view answer</summary>

<button onclick="copyAnswer1()">Copy to clipboard</button>

<pre id="answer-1" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">

<code>
test_query_2 = "What would be the estimated fare for a 5.2 mile trip with a base fare of $3.50?"
result_2 = agent.predict(agent_payload(test_query_2))
print(f"Query: {test_query_2}")
print(f"Response: {result_2}")
</code></pre>

</details>

<script>
function copyAnswer1() {
 const el = document.getElementById("answer-1");
 if (!el) return;


 const text = el.innerText;


 if (navigator.clipboard && navigator.clipboard.writeText) {
   navigator.clipboard.writeText(text)
     .then(() => alert("Copied to clipboard"))
     .catch(err => {
       console.error("Clipboard write failed:", err);
       fallbackCopy(text);
     });
 } else {
   fallbackCopy(text);
 }
}


function fallbackCopy(text) {
 const textarea = document.createElement("textarea");
 textarea.value = text;
 textarea.style.position = "fixed";
 textarea.style.left = "-9999px";
 document.body.appendChild(textarea);
 textarea.select();
 try {
   document.execCommand("copy");
   alert("Copied to clipboard");
 } catch (err) {
   console.error("Fallback copy failed:", err);
   alert("Could not copy to clipboard. Please copy manually.");
 } finally {
   document.body.removeChild(textarea);
 }
}
</script>


## Part 2. Built-In Judges Implementation

### 2.1. Create Evaluation Dataset for Built-In Judges

Create an evaluation dataset with two inputs, expected facts, and expected responses for testing correctness and safety judges. Part of the evaluation dataset has been filled out for you.

**Hint:** If you get stuck, you can view a sample evaluation dataset in `./artifacts/evaluation_datasets/nyc_taxi_eval.json`.

In [0]:
## Create an evaluation dataset for built-in judges
## Include inputs and expectations with expected_facts for each query
eval_dataset_builtin = [
  {
    "inputs": {
      "input": [
        {
          "role": <FILL_IN>,
          "content": <FILL_IN>
        }
      ]
    },
    "expectations": {
      "expected_facts": [
        <FILL_IN>
      ]
    }
  },
  {
    "inputs": {
      "input": [
        {
          "role": <FILL_IN>,
          "content": <FILL_IN>
        }
      ]
    },
    "expectations": {
      "expected_facts": [
        <FILL_IN>
      ]
    }
  }
]

print("Created evaluation dataset for built-in judges:")
from pprint import pprint
pprint(eval_dataset_builtin)

<details>
<summary style="cursor: pointer; padding: 10px; border: 1px solid #ccc; background-color: #f9f9f9; border-radius: 5px; font-weight: bold;">Click to view answer</summary>

<button onclick="copyAnswer1()">Copy to clipboard</button>

<pre id="answer-1" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">



<code>
eval_dataset_builtin = [
  {
    "inputs": {
      "input": [
        {
          "role": "user",
          "content": "What is the average fare for trips from zip code 10001 to 10002?"
        }
      ]
    },
    "expectations": {
      "expected_facts": [
        "Tool used is trip_info",
        "Pickup zip code is 10001",
        "Dropoff zip code is 10002",
        "Response includes average fare information"
      ]
    }
  },
  {
    "inputs": {
      "input": [
        {
          "role": "user",
          "content": "How many trips were taken from zip code 10003 to 10004?"
        }
      ]
    },
    "expectations": {
      "expected_facts": [
        "Tool used is trip_info",
        "Pickup zip code is 10003",
        "Dropoff zip code is 10004",
        "Response includes trip count",
        "Response includes 'Are there any additional questions I can help with?'"
      ]
    }
  }
]

print("Created evaluation dataset for built-in judges:")
from pprint import pprint
pprint(eval_dataset_builtin)
</code></pre>

</details>

<script>
function copyAnswer1() {
 const el = document.getElementById("answer-1");
 if (!el) return;


 const text = el.innerText;


 if (navigator.clipboard && navigator.clipboard.writeText) {
   navigator.clipboard.writeText(text)
     .then(() => alert("Copied to clipboard"))
     .catch(err => {
       console.error("Clipboard write failed:", err);
       fallbackCopy(text);
     });
 } else {
   fallbackCopy(text);
 }
}


function fallbackCopy(text) {
 const textarea = document.createElement("textarea");
 textarea.value = text;
 textarea.style.position = "fixed";
 textarea.style.left = "-9999px";
 document.body.appendChild(textarea);
 textarea.select();
 try {
   document.execCommand("copy");
   alert("Copied to clipboard");
 } catch (err) {
   console.error("Fallback copy failed:", err);
   alert("Could not copy to clipboard. Please copy manually.");
 } finally {
   document.body.removeChild(textarea);
 }
}
</script>

### 2.2. Configure Built-In Judges

Set up the Correctness and Safety judges using the configured endpoints.

In [0]:
## Import and configure the built-in judges
from mlflow.genai.scorers import <FILL_IN>, <FILL_IN>

## Create correctness judge instance
correctness_eval = <FILL_IN>(
    model=<FILL_IN>
)

## Create safety judge instance
safety_eval = <FILL_IN>(
    model=<FILL_IN>
)

<details>
<summary style="cursor: pointer; padding: 10px; border: 1px solid #ccc; background-color: #f9f9f9; border-radius: 5px; font-weight: bold;">Click to view answer</summary>


<button onclick="copyAnswer1()">Copy to clipboard</button>


<pre id="answer-1" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">



<code>
from mlflow.genai.scorers import Correctness, Safety

# Create correctness judge instance
correctness_eval = Correctness(
    model=correctness_eval_endpoint
)

# Create safety judge instance
safety_eval = Safety(
    model=safety_endpoint
)
</code></pre>

</details>

<script>
function copyAnswer1() {
 const el = document.getElementById("answer-1");
 if (!el) return;


 const text = el.innerText;


 if (navigator.clipboard && navigator.clipboard.writeText) {
   navigator.clipboard.writeText(text)
     .then(() => alert("Copied to clipboard"))
     .catch(err => {
       console.error("Clipboard write failed:", err);
       fallbackCopy(text);
     });
 } else {
   fallbackCopy(text);
 }
}


function fallbackCopy(text) {
 const textarea = document.createElement("textarea");
 textarea.value = text;
 textarea.style.position = "fixed";
 textarea.style.left = "-9999px";
 document.body.appendChild(textarea);
 textarea.select();
 try {
   document.execCommand("copy");
   alert("Copied to clipboard");
 } catch (err) {
   console.error("Fallback copy failed:", err);
   alert("Could not copy to clipboard. Please copy manually.");
 } finally {
   document.body.removeChild(textarea);
 }
}
</script>


### 2.3. Execute Built-In Judge Evaluation

Run the evaluation using both correctness and safety judges simultaneously. Click on the **View evaluation results with MLflow** to inspect the results.

In [0]:
## Execute evaluation with both built-in judges
builtin_results = mlflow.genai.evaluate(
    data=<FILL_IN>,
    predict_fn=<FILL_IN>,
    scorers=<FILL_IN>
)

<details>
<summary style="cursor: pointer; padding: 10px; border: 1px solid #ccc; background-color: #f9f9f9; border-radius: 5px; font-weight: bold;">Click to view answer</summary>


<button onclick="copyAnswer1()">Copy to clipboard</button>


<pre id="answer-1" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">


<code>
builtin_results = mlflow.genai.evaluate(
    data=eval_dataset_builtin,
    predict_fn=lambda input: agent.predict({"input": input}),
    scorers=[correctness_eval, safety_eval]
)
</code></pre>

</details>

<script>
function copyAnswer1() {
 const el = document.getElementById("answer-1");
 if (!el) return;


 const text = el.innerText;


 if (navigator.clipboard && navigator.clipboard.writeText) {
   navigator.clipboard.writeText(text)
     .then(() => alert("Copied to clipboard"))
     .catch(err => {
       console.error("Clipboard write failed:", err);
       fallbackCopy(text);
     });
 } else {
   fallbackCopy(text);
 }
}


function fallbackCopy(text) {
 const textarea = document.createElement("textarea");
 textarea.value = text;
 textarea.style.position = "fixed";
 textarea.style.left = "-9999px";
 document.body.appendChild(textarea);
 textarea.select();
 try {
   document.execCommand("copy");
   alert("Copied to clipboard");
 } catch (err) {
   console.error("Fallback copy failed:", err);
   alert("Could not copy to clipboard. Please copy manually.");
 } finally {
   document.body.removeChild(textarea);
 }
}
</script>

### 2.4. Analyze Built-In Judge Results

Examine the evaluation results and understand the scoring rationale using print statements and `builtin_results()`.

In [0]:
## Display the evaluation results and metrics
print(f"Run ID: {<FILL_IN>}")
print(f"Aggregated Metrics: {<FILL_IN>}")
print("\nDetailed Results:")
display(<FILL_IN>)

<details>
<summary style="cursor: pointer; padding: 10px; border: 1px solid #ccc; background-color: #f9f9f9; border-radius: 5px; font-weight: bold;">Click to view answer</summary>


<button onclick="copyAnswer1()">Copy to clipboard</button>


<pre id="answer-1" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">


<code>
print(f"Run ID: {builtin_results.run_id}")
print(f"Aggregated Metrics: {builtin_results.metrics}")
print("\nDetailed Results:")
display(builtin_results.result_df)
</code></pre>

</details>

<script>
function copyAnswer1() {
 const el = document.getElementById("answer-1");
 if (!el) return;


 const text = el.innerText;


 if (navigator.clipboard && navigator.clipboard.writeText) {
   navigator.clipboard.writeText(text)
     .then(() => alert("Copied to clipboard"))
     .catch(err => {
       console.error("Clipboard write failed:", err);
       fallbackCopy(text);
     });
 } else {
   fallbackCopy(text);
 }
}


function fallbackCopy(text) {
 const textarea = document.createElement("textarea");
 textarea.value = text;
 textarea.style.position = "fixed";
 textarea.style.left = "-9999px";
 document.body.appendChild(textarea);
 textarea.select();
 try {
   document.execCommand("copy");
   alert("Copied to clipboard");
 } catch (err) {
   console.error("Fallback copy failed:", err);
   alert("Could not copy to clipboard. Please copy manually.");
 } finally {
   document.body.removeChild(textarea);
 }
}
</script>

### 2.5. Update and Iterate (Optional)

Did your agent pass your expectations? If not, try updating the system prompt in `./artifacts/configs/nyc_taxi_agent_config.yaml` so that the agent aligns with your evaluation expectations. Iterate on top of this process all evaluations **Pass**. 

For example, the default system prompt is 

```
You are an expert in answering questions regarding NYC Taxi data.
```

Which isn't very descriptive if, for example, we pass the question _What is the average fare for trips from zip code 10001 to 10002?_ and expect the following in our output 
- Tool used is trip_info,
- Pickup zip code is 10001,
- Dropoff zip code is 10002,
- Response includes 'Are there any additional questions I can help with?'

then we might update our system prompt with _You are an expert in answering questions regarding NYC Taxi data. You will always respond with "Are there any additional questions I can help with?"_ (see screenshot below).

![update_system_prompt.png](../Includes/images/applying agent eval/update_system_prompt.png "update_system_prompt.png")

After updating the prompt, you can redeploy your agent with the following code snippet.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
import mlflow
import yaml
from mlflow.models.resources import DatabricksServingEndpoint, DatabricksFunction

# Point to your existing YAML config file (the same one you pass as model_config)
CONFIG_YAML = "../artifacts/configs/nyc_taxi_agent_config.yaml"  # &lt;— set this

# Load config from YAML
with open(CONFIG_YAML, "r") as f:
    cfg = yaml.safe_load(f)

# Pull values from YAML (with sensible fallbacks)
CATALOG = cfg["CATALOG_NAME"]
SCHEMA = cfg["SCHEMA_NAME"]
MODEL_BASE_NAME = cfg.get("MODEL_NAME", "nyc_taxi_eval_agent")  # optional key in YAML; defaults to "nyc_agent"
UC_MODEL_NAME = f"{CATALOG}.{SCHEMA}.{MODEL_BASE_NAME}"

LLM_ENDPOINT = cfg["LLM_ENDPOINT_NAME"]

# Tools can be provided either as TOOL1/TOOL2, or as a TOOLS: [ ... ] list in YAML
if "TOOLS" in cfg and isinstance(cfg["TOOLS"], (list, tuple)) and len(cfg["TOOLS"]) &gt; 0:
    tool_names = [f"{CATALOG}.{SCHEMA}.{t}" for t in cfg["TOOLS"]]
else:
    tool1 = cfg.get("TOOL1")
    tool2 = cfg.get("TOOL2")
    tool_names = [f"{CATALOG}.{SCHEMA}.{t}" for t in [tool1, tool2] if t]

# Build MLflow resources from YAML-driven values
resources = [DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT)] + [
    DatabricksFunction(function_name=t) for t in tool_names
]

# Same python model file you used before
AGENT_PY = "../artifacts/nyc_taxi_agent.py"

# Optional alias (could also come from YAML via cfg.get("ALIAS", "champion"))
ALIAS = "challenger"

mlflow.set_registry_uri("databricks-uc")  # ensure we register to UC

with mlflow.start_run(tags={
    "training_type": "agent_eval_training",
    "model": LLM_ENDPOINT,
    "agent_type": "TOOL-CALLING",
    "agent_id": "agent2",
}):
    logged = mlflow.pyfunc.log_model(
        artifact_path="agent",
        python_model=AGENT_PY,
        model_config=CONFIG_YAML,  # the YAML you just edited (includes SYSTEM_PROMPT)
        artifacts={
            "agent_config": CONFIG_YAML,
            # "agent_eval_config": "/Workspace/Users/you/path/to/eval_config.yaml",  # optional
        },
        input_example={"input": [{"role": "user", "content": "hello"}]},
        pip_requirements=[
            "databricks-openai", "backoff", "pyyaml",
            # If you previously pinned databricks-connect, keep your pin here:
            # f"databricks-connect=={version('databricks-connect')}",
        ],
        resources=resources,
        registered_model_name=UC_MODEL_NAME,  # creates a new UC model version
    )

new_version = logged.registered_model_version
print("New UC version:", new_version)

# Optionally flip your alias to the new version:
from mlflow.tracking import MlflowClient
MlflowClient().set_registered_model_alias(
    name=UC_MODEL_NAME, alias=ALIAS, version=new_version
)

# Load by alias
agent = mlflow.pyfunc.load_model(f"models:/{UC_MODEL_NAME}@{ALIAS}")
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

You can now go back to sections **2.3** and **2.4** and re-evaluate.

## Part 3. Guideline Judges Implementation

### 3.1. Create Global Guidelines Dataset

Create an evaluation dataset for testing global guidelines that apply uniformly to all responses.

**Hint:** If you get stuck, you can view a sample evaluation dataset in `./artifacts/evaluation_datasets/nyc_taxi_guidelines_eval.json`.

In [0]:
## Create evaluation dataset for global guidelines
eval_dataset_guidelines = [
    {
        "inputs": <FILL_IN>
    }
]

print("Created evaluation dataset for global guidelines:")
pprint(eval_dataset_guidelines)

<details>
<summary style="cursor: pointer; padding: 10px; border: 1px solid #ccc; background-color: #f9f9f9; border-radius: 5px; font-weight: bold;">Click to view answer</summary>


<button onclick="copyAnswer1()">Copy to clipboard</button>


<pre id="answer-1" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">


<code>
eval_dataset_guidelines = [
  {
    "inputs": {
      "input": [
        {
          "role": "user",
          "content": "What is the average fare for trips from zip code 10001 to 10002?"
        }
      ]
    },
    "expectations": {
      "expected_facts": [
        "The average fare for trips between zip codes 10001 and 10002 is $12.50."
      ],
      "guidelines": [
        "The response should be professional and courteous",
        "The response should include specific numerical values when providing calculations or statistics",
        "The response should clearly indicate when it's using NYC taxi data"
    ]
    }
  }
]

print("Created evaluation dataset for global guidelines:")
pprint(eval_dataset_guidelines)
</code></pre>

</details>

<script>
function copyAnswer1() {
 const el = document.getElementById("answer-1");
 if (!el) return;


 const text = el.innerText;


 if (navigator.clipboard && navigator.clipboard.writeText) {
   navigator.clipboard.writeText(text)
     .then(() => alert("Copied to clipboard"))
     .catch(err => {
       console.error("Clipboard write failed:", err);
       fallbackCopy(text);
     });
 } else {
   fallbackCopy(text);
 }
}


function fallbackCopy(text) {
 const textarea = document.createElement("textarea");
 textarea.value = text;
 textarea.style.position = "fixed";
 textarea.style.left = "-9999px";
 document.body.appendChild(textarea);
 textarea.select();
 try {
   document.execCommand("copy");
   alert("Copied to clipboard");
 } catch (err) {
   console.error("Fallback copy failed:", err);
   alert("Could not copy to clipboard. Please copy manually.");
 } finally {
   document.body.removeChild(textarea);
 }
}
</script>


### 3.2. Implement Global Guidelines Judge

Create a global guidelines judge that ensures responses are professional and include specific formatting requirements.

In [0]:
## Import and configure the Guidelines judge
from mlflow.genai.scorers import <FILL_IN>

## Create a global guideline for professional responses
professional_guideline = <FILL_IN>(
    name=<FILL_IN>,
    guidelines=[
        <FILL_IN>,
        <FILL_IN>
    ],
    model_name=<FILL_IN>
)

<details>
<summary style="cursor: pointer; padding: 10px; border: 1px solid #ccc; background-color: #f9f9f9; border-radius: 5px; font-weight: bold;">Click to view answer</summary>

<button onclick="copyAnswer1()">Copy to clipboard</button>

<pre id="answer-1" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">

<code>
from mlflow.genai.scorers import Guidelines

# Create a global guideline for professional responses
professional_guideline = Guidelines(
    name="professional_response",
    guidelines=[
        "The response should be professional and courteous",
        "The response should include specific numerical values when providing calculations or statistics",
        "The response should clearly indicate when it's using NYC taxi data"
    ],
    model_name=guidelines_endpoint
)
</code></pre>

</details>

<script>
function copyAnswer1() {
 const el = document.getElementById("answer-1");
 if (!el) return;


 const text = el.innerText;


 if (navigator.clipboard && navigator.clipboard.writeText) {
   navigator.clipboard.writeText(text)
     .then(() => alert("Copied to clipboard"))
     .catch(err => {
       console.error("Clipboard write failed:", err);
       fallbackCopy(text);
     });
 } else {
   fallbackCopy(text);
 }
}


function fallbackCopy(text) {
 const textarea = document.createElement("textarea");
 textarea.value = text;
 textarea.style.position = "fixed";
 textarea.style.left = "-9999px";
 document.body.appendChild(textarea);
 textarea.select();
 try {
   document.execCommand("copy");
   alert("Copied to clipboard");
 } catch (err) {
   console.error("Fallback copy failed:", err);
   alert("Could not copy to clipboard. Please copy manually.");
 } finally {
   document.body.removeChild(textarea);
 }
}
</script>

### 3.3. Execute Global Guidelines Evaluation

Run the evaluation using the global guidelines judge. Click on the **View evaluation results with MLflow** to inspect the results.

In [0]:
## Execute evaluation with global guidelines
guidelines_results = mlflow.genai.evaluate(
    data=<FILL_IN>,
    predict_fn=<FILL_IN>,
    scorers=<FILL_IN>
)

<details>
<summary style="cursor: pointer; padding: 10px; border: 1px solid #ccc; background-color: #f9f9f9; border-radius: 5px; font-weight: bold;">Click to view answer</summary>


<button onclick="copyAnswer1()">Copy to clipboard</button>

<pre id="answer-1" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">


<code>
guidelines_results = mlflow.genai.evaluate(
    data=eval_dataset_guidelines,
    predict_fn=lambda input: agent.predict({"input": input}),
    scorers=[professional_guideline]
)
</code>

</pre>

</details>

<script>
function copyAnswer1() {
 const el = document.getElementById("answer-1");
 if (!el) return;


 const text = el.innerText;


 if (navigator.clipboard && navigator.clipboard.writeText) {
   navigator.clipboard.writeText(text)
     .then(() => alert("Copied to clipboard"))
     .catch(err => {
       console.error("Clipboard write failed:", err);
       fallbackCopy(text);
     });
 } else {
   fallbackCopy(text);
 }
}


function fallbackCopy(text) {
 const textarea = document.createElement("textarea");
 textarea.value = text;
 textarea.style.position = "fixed";
 textarea.style.left = "-9999px";
 document.body.appendChild(textarea);
 textarea.select();
 try {
   document.execCommand("copy");
   alert("Copied to clipboard");
 } catch (err) {
   console.error("Fallback copy failed:", err);
   alert("Could not copy to clipboard. Please copy manually.");
 } finally {
   document.body.removeChild(textarea);
 }
}
</script>

## Part 4. Comprehensive Evaluation Analysis

### 4.1. Compare Evaluation Results

Analyze and compare results from different judge types to understand their strengths and use cases.

In [0]:
## Display results from all evaluation runs for comparison
print("=== BUILT-IN JUDGES RESULTS ===")
print(f"Run ID: {<FILL_IN>}")
print(f"Metrics: {<FILL_IN>}")
display(<FILL_IN>)

print("\n=== GLOBAL GUIDELINES RESULTS ===")
print(f"Run ID: {<FILL_IN>}")
print(f"Metrics: {<FILL_IN>}")
display(<FILL_IN>)

<details>
<summary style="cursor: pointer; padding: 10px; border: 1px solid #ccc; background-color: #f9f9f9; border-radius: 5px; font-weight: bold;">Click to view answer</summary>

<button onclick="copyAnswer1()">Copy to clipboard</button>

<pre id="answer-1" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
print("=== BUILT-IN JUDGES RESULTS ===")
print(f"Run ID: {builtin_results.run_id}")
print(f"Metrics: {builtin_results.metrics}")
display(builtin_results.result_df)

print("\n=== GLOBAL GUIDELINES RESULTS ===")
print(f"Run ID: {guidelines_results.run_id}")
print(f"Metrics: {guidelines_results.metrics}")
display(guidelines_results.result_df)
</code></pre>

</details>

<script>
function copyAnswer1() {
 const el = document.getElementById("answer-1");
 if (!el) return;


 const text = el.innerText;


 if (navigator.clipboard && navigator.clipboard.writeText) {
   navigator.clipboard.writeText(text)
     .then(() => alert("Copied to clipboard"))
     .catch(err => {
       console.error("Clipboard write failed:", err);
       fallbackCopy(text);
     });
 } else {
   fallbackCopy(text);
 }
}


function fallbackCopy(text) {
 const textarea = document.createElement("textarea");
 textarea.value = text;
 textarea.style.position = "fixed";
 textarea.style.left = "-9999px";
 document.body.appendChild(textarea);
 textarea.select();
 try {
   document.execCommand("copy");
   alert("Copied to clipboard");
 } catch (err) {
   console.error("Fallback copy failed:", err);
   alert("Could not copy to clipboard. Please copy manually.");
 } finally {
   document.body.removeChild(textarea);
 }
}
</script>

### 4.2. Create Comprehensive Evaluation

Combine multiple judge types in a single evaluation run to get comprehensive quality assessment.

In [0]:
## Create a comprehensive evaluation using multiple judge types
## Use the built-in dataset and combine correctness with global guidelines
comprehensive_results = mlflow.genai.evaluate(
    data=<FILL_IN>,
    predict_fn=<FILL_IN>,
    scorers=<FILL_IN>
)

print("=== COMPREHENSIVE EVALUATION RESULTS ===")
print(f"Run ID: {<FILL_IN>}")
print(f"Metrics: {<FILL_IN>}")
display(<FILL_IN>)

<details>
<summary style="cursor: pointer; padding: 10px; border: 1px solid #ccc; background-color: #f9f9f9; border-radius: 5px; font-weight: bold;">Click to view answer</summary>


<button onclick="copyAnswer1()">Copy to clipboard</button>


<pre id="answer-1" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">

<pre id="answer-13" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
comprehensive_results = mlflow.genai.evaluate(
    data=eval_dataset_builtin,
    predict_fn=lambda input: agent.predict({"input": input}),
    scorers=[correctness_eval, professional_guideline]
)

print("=== COMPREHENSIVE EVALUATION RESULTS ===")
print(f"Run ID: {comprehensive_results.run_id}")
print(f"Metrics: {comprehensive_results.metrics}")
display(comprehensive_results.result_df)
</code></pre>

</details>

<script>
function copyAnswer1() {
 const el = document.getElementById("answer-1");
 if (!el) return;


 const text = el.innerText;


 if (navigator.clipboard && navigator.clipboard.writeText) {
   navigator.clipboard.writeText(text)
     .then(() => alert("Copied to clipboard"))
     .catch(err => {
       console.error("Clipboard write failed:", err);
       fallbackCopy(text);
     });
 } else {
   fallbackCopy(text);
 }
}


function fallbackCopy(text) {
 const textarea = document.createElement("textarea");
 textarea.value = text;
 textarea.style.position = "fixed";
 textarea.style.left = "-9999px";
 document.body.appendChild(textarea);
 textarea.select();
 try {
   document.execCommand("copy");
   alert("Copied to clipboard");
 } catch (err) {
   console.error("Fallback copy failed:", err);
   alert("Could not copy to clipboard. Please copy manually.");
 } finally {
   document.body.removeChild(textarea);
 }
}
</script>


## Conclusion

In this lab, you have successfully implemented a comprehensive evaluation framework for AI agents using MLflow's built-in and guideline judges. You learned how to:

- Configure and execute multiple types of judges for different evaluation needs
- Create properly structured evaluation datasets for various judge types
- Analyze evaluation results using MLflow's tracing and result analysis capabilities
- Apply best practices for evaluation configuration and model management

These evaluation techniques provide the foundation for ensuring your AI agents meet quality standards and perform reliably in production environments. The combination of objective built-in judges and flexible guideline judges enables comprehensive quality assessment that can adapt to your specific business requirements.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>